:::{admonition} Download
:class: important

Download this notebook: **{nb-download}`population_classifier.ipynb`**!

:::


(glm-population-classifier)=
# Population Classifier GLM

[`ClassifierPopulationGLM`](nemos.glm.ClassifierPopulationGLM) predicts discrete labels for several outputs at once: one categorical variable for each neuron. It uses a softmax inverse link. It is to the [`ClassifierGLM`](nemos.glm.ClassifierGLM) what the [`PopulationGLM`](nemos.glm.PopulationGLM) is to the [`GLM`](nemos.glm.GLM). It is the same model, fit jointly over the neuron axis instead of in a loop. The difference is that axis: `classes`, the [class labels](classifier-labels) and the [identifiability of the parameters](classifier-identifiability) work as in the [classifier GLM](glm-classifier).

```{image} ../../../assets/population_classifier_glm.svg
:alt: ClassifierPopulationGLM. X times a weight tensor W with one group of class columns per neuron, then a softmax gives the probabilities P of shape (n_samples, n_neurons, n_classes), and the argmax gives one label per sample and neuron.
:class: only-light
```

```{image} ../../../assets/population_classifier_glm_dark.svg
:alt: ClassifierPopulationGLM. X times a weight tensor W with one group of class columns per neuron, then a softmax gives the probabilities P of shape (n_samples, n_neurons, n_classes), and the argmax gives one label per sample and neuron.
:class: only-dark
```

```{contents}
:local:
:depth: 3
```

## Initializing a ClassifierPopulationGLM object

All the neurons share one set of classes. Declare them with `classes` at initialization, as an integer or as the array of labels. As for the [`PopulationGLM`](nemos.glm.PopulationGLM), a [`feature_mask`](neuron-specific-features-classifier) selects the predictors of each neuron.

In [1]:
import jax.numpy as jnp
import numpy as np

import nemos as nmo

n_neurons = 3
n_classes = 4
n_samples = 1000

nmo.glm.ClassifierPopulationGLM(classes=n_classes)

,classes,4
,inverse_link_function,<function log...x7feac4c28a40>
,regularizer,Ridge()
,regularizer_strength,1.0
,fix_params,"(None, ...)"
,solver_name,'Newton'
,solver_kwargs,{}
,fit_intercept,True
,feature_mask,None
Name,Type,Value
aux_,NoneType,None


## Inputs

`X` is the same as for the [single neuron GLM](glm-input-types). `y` contains the labels, with one column for each neuron.

### Arrays

`X` has shape `(n_samples, n_features)` and `y` has shape `(n_samples, n_neurons)`. Each of them can be a NumPy array, a JAX array, or a [pynapple](https://www.pynapple.org) time series.

#### NumPy

In [2]:
# X of shape (n_samples, n_features), labels of shape (n_samples, n_neurons)
X, labels = nmo.datasets.tuned_categorical(
    n_samples=n_samples, n_neurons=n_neurons, n_classes=n_classes
)  # JAX arrays
X_np, labels_np = np.asarray(X), np.asarray(labels, dtype=int)

print(f"X shape:      {X.shape}")
print(f"labels shape: {labels.shape}")

model_np = nmo.glm.ClassifierPopulationGLM(classes=n_classes).fit(X_np, labels_np)

X shape:      (1000, 32)
labels shape: (1000, 3)


#### JAX

`fit` gives the same parameters for JAX arrays and for NumPy arrays.

In [3]:
model_jax = nmo.glm.ClassifierPopulationGLM(classes=n_classes).fit(X, labels)

np.allclose(model_jax.coef_, model_np.coef_)

True

#### Pynapple

With [pynapple](https://www.pynapple.org), `y` is a `TsdFrame` with one column for each neuron. With [pynapple](https://www.pynapple.org) predictors, `predict` returns a `TsdFrame` of labels with the column names of `y`. `predict_proba` returns a `TsdTensor` with shape `(n_samples, n_neurons, n_classes)`.

In [4]:
import pynapple as nap

# suppress jax to numpy conversion warning
nap.nap_config.suppress_conversion_warnings = True

# 10 ms bins
time = np.arange(n_samples) / 100

predictors = nap.TsdFrame(t=time, d=X_np)
labels_nap = nap.TsdFrame(t=time, d=labels_np, columns=["unit0", "unit1", "unit2"])

model_nap = nmo.glm.ClassifierPopulationGLM(classes=n_classes).fit(
    predictors, labels_nap
)
model_nap.predict(predictors)

Time (s)      unit0    unit1    unit2
----------  -------  -------  -------
0.0               1        3        0
0.01              1        3        0
0.02              1        3        0
0.03              1        3        0
0.04              1        1        0
0.05              1        3        0
0.06              1        3        0
...
9.93              1        3        0
9.94              1        1        0
9.95              1        3        0
9.96              1        3        0
9.97              1        3        0
9.98              1        3        0
9.99              1        3        0
dtype: int32, shape: (1000, 3)

### PyTrees

If `X` is a PyTree, each leaf of `coef_` has the shape of its predictors, plus an axis for the neurons and an axis for the classes.

In [5]:
X_dict = {"stimulus": X[:, :12], "coupling": X[:, 12:]}

model_dict = nmo.glm.ClassifierPopulationGLM(classes=n_classes).fit(X_dict, labels)

{key: coef.shape for key, coef in model_dict.coef_.items()}

{'coupling': (20, 3, 4), 'stimulus': (12, 3, 4)}

## Methods

### Fit

In [6]:
model = nmo.glm.ClassifierPopulationGLM(classes=n_classes).fit(X, labels)
model

,classes,4
,inverse_link_function,<function log...x7feac4c28a40>
,regularizer,Ridge()
,regularizer_strength,1.0
,fix_params,"(None, ...)"
,solver_name,'Newton'
,solver_kwargs,{}
,fit_intercept,True
,feature_mask,None
Name,Type,Value
aux_,NoneType,None


#### Fitted attributes

The model estimates one set of coefficients for each class. Thus, the parameters have an additional last axis for the classes.

In [7]:
print(f"coef_ shape:      {model.coef_.shape}")
print(f"intercept_ shape: {model.intercept_.shape}")

coef_ shape:      (32, 3, 4)
intercept_ shape: (3, 4)


- `coef_` has shape `(n_features, n_neurons, n_classes)`. `coef_[:, j, k]` holds the weights of neuron `j` for class `k`.
- `intercept_` has shape `(n_neurons, n_classes)`.

Without the class axis, these are the shapes of the [`PopulationGLM`](nemos.glm.PopulationGLM). Without the neuron axis too, they are the shapes of the single neuron [`ClassifierGLM`](nemos.glm.ClassifierGLM): `(n_features, n_classes)` and `(n_classes,)`.

The class axis has the same over-parameterization as in the single neuron model: one linear predictor for each class, for probabilities that sum to one. Thus, `Ridge` is also the default here. See [identifiability](classifier-identifiability) for which parameters the penalty determines and which it leaves free.

#### NaN treatment

The fit ignores samples with a NaN in `X` or `y`. Examples of such samples are bins without a measurement, and the edges that a convolution leaves undefined. You do not need to remove these samples by hand.

In [8]:
X_nan = X.at[:10].set(np.nan)

model_nan = nmo.glm.ClassifierPopulationGLM(classes=n_classes).fit(X_nan, labels)
model_dropped = nmo.glm.ClassifierPopulationGLM(classes=n_classes).fit(
    X[10:], labels[10:]
)

np.allclose(model_nan.coef_, model_dropped.coef_)

True

### Predict

[`predict`](nemos.glm.ClassifierPopulationGLM.predict) takes the arg-max over the class axis. The output has one label for each sample and neuron.

In [9]:
predicted = model.predict(X)
print(f"predicted shape: {predicted.shape}")
predicted[:5]

predicted shape: (1000, 3)


Array([[1, 3, 0],
       [1, 3, 0],
       [1, 3, 0],
       [1, 3, 0],
       [1, 1, 0]], dtype=int32)

[`predict_proba`](nemos.glm.ClassifierPopulationGLM.predict_proba) keeps the class axis. The output has the distribution of every neuron along this axis.

In [10]:
proba = model.predict_proba(X, return_type="proba")
print(f"proba shape: {proba.shape}")

# the distribution of each neuron at the first sample
proba[0]

proba shape: (1000, 3, 4)


Array([[0.2472172 , 0.27908528, 0.23097086, 0.2427267 ],
       [0.24178259, 0.2553942 , 0.24396594, 0.25885737],
       [0.26758692, 0.25076184, 0.22735205, 0.2542992 ]], dtype=float32)

### Score

[`score`](nemos.glm.ClassifierPopulationGLM.score) returns the mean log-likelihood over samples and neurons, as a single number. scikit-learn cross-validation expects this from an estimator.

In [11]:
model.score(X, labels)

Array(-1.3830429, dtype=float32)

As for the [`PopulationGLM`](nemos.glm.PopulationGLM), an average over the sample axis only gives one score per neuron.

In [12]:
model.score(X, labels, aggregate_sample_scores=lambda x: jnp.mean(x, axis=0))

Array([-1.3818685, -1.3842758, -1.3829851], dtype=float32)

### Simulate

[`simulate`](nemos.glm.ClassifierPopulationGLM.simulate) draws a label for each sample and each neuron from the predicted class distribution. It returns the labels and the log-probabilities they were drawn from.

In [13]:
import jax

labels_sim, log_proba_sim = model.simulate(jax.random.key(0), X)

print(f"labels:     {labels_sim.shape}")
print(f"log-probas: {log_proba_sim.shape}")

labels:     (1000, 3)
log-probas: (1000, 3, 4)


The random key makes the draw reproducible: the same key and the same probabilities give the same labels.

(population-classifier-out-of-memory)=
### Fitting out of memory

If the design matrix does not fit in memory, use [`stochastic_fit`](nemos.glm.ClassifierPopulationGLM.stochastic_fit). It takes a data loader instead of arrays, and updates the parameters one batch at a time. Each batch of `y` contains all the neurons. Stochastic solvers require acceleration and line searches to be off.

A single batch does not always contain every class, so declare the labels with [`set_classes`](nemos.glm.ClassifierPopulationGLM.set_classes) before you start.

In [14]:
# the loader serves the data in batches of 100 samples
loader = nmo.batching.ArrayDataLoader(X, labels, batch_size=100)

clf = nmo.glm.ClassifierPopulationGLM(
    classes=n_classes,
    solver_name="GradientDescent",
    solver_kwargs={"stepsize": 0.05, "acceleration": False},
).set_classes(labels)

clf.stochastic_fit(loader, n_passes=20)

,classes,"Array([0., 1....dtype=float32)"
,inverse_link_function,<function log...x7feac4c28a40>
,regularizer,Ridge()
,regularizer_strength,1.0
,fix_params,"(None, ...)"
,solver_name,'GradientDescent'
,solver_kwargs,"{'acceleration': False, 'stepsize': 0.05}"
,fit_intercept,True
,feature_mask,None
Name,Type,Value
aux_,NoneType,None


See [stochastic optimization](../../scalability/README.md).

#### Update

For full control of the optimization loop, call [`update`](nemos.glm.ClassifierPopulationGLM.update) to do each step yourself. [`initialize_params`](nemos.glm.ClassifierPopulationGLM.initialize_params) returns the initial parameters. [`initialize_optimizer_and_state`](nemos.glm.ClassifierPopulationGLM.initialize_optimizer_and_state) sets up the solver.

In [15]:
params = clf.initialize_params(X, labels)
opt_state = clf.initialize_optimizer_and_state(params, X, labels)

# a full pass over all the batches
for X_batch, labels_batch in loader:
    params, opt_state = clf.update(params, opt_state, X_batch, labels_batch)

The loop body can contain any code. Examples are a log of the score for each batch, a change of the learning rate between passes, or your own stopping rule. See [custom update](../../scalability/custom_update.md).

(population-classifier-model-configuration)=
## Model configurations

### Observation models

The observation model is fixed to [`CategoricalObservations`](nemos.observation_models.CategoricalObservations), and it is not an argument of the constructor. Every neuron shares it.

### Inverse link function

The inverse link function maps the linear term to the log-probability of each label, and the default is `log_softmax`, which produces it directly. It is the only choice that keeps the log-likelihood convex.

### Regularizers

Set the regularizer and its strength when you create the model.

In [16]:
nmo.glm.ClassifierPopulationGLM(
    classes=n_classes, regularizer="Lasso", regularizer_strength=0.05
).fit(X, labels).coef_.shape

(32, 3, 4)

As for the [classifier GLM](glm-classifier), the default is `Ridge` with a strength of 1 rather than no regularization, because the class axis is over-parameterized. A parameter-wise `regularizer_strength` has the shape of the parameters, including the neuron and class axes.

[Regularizers](../../regularizers.md) describes the group-wise and parameter-wise forms, and the special cases of ElasticNet and GroupLasso.

### Solvers

Select the solver by name, and configure it with `solver_kwargs`. The whole population is fit in a single optimization.

In [17]:
print(nmo.glm.ClassifierPopulationGLM(classes=n_classes).solver_name)
print(
    nmo.glm.ClassifierPopulationGLM(
        classes=n_classes, regularizer="Lasso", regularizer_strength=0.01
    ).solver_name
)

Newton
ProximalGradient


[Solvers](../../solvers.md) describes which solver accepts which regularizer, which solver to use, and how to register your own solver.

## Parameter specification

### Drop the intercept

`fit_intercept=False` removes the intercept from the model. It stays at zero for every neuron and class, and uses no degree of freedom.

In [18]:
no_intercept = nmo.glm.ClassifierPopulationGLM(
    classes=n_classes, fit_intercept=False
).fit(X, labels)

print(f"intercept_: {no_intercept.intercept_.shape}")
print(f"dof_resid_: {no_intercept.dof_resid_}")

intercept_: (3, 4)
dof_resid_: [904. 904. 904.]


### Fix parameter values

`fix_params` keeps the intercept at a value that you choose. It has the shape of `intercept_`, one value for each neuron and class. Give `None` for the leaf to learn.

In [19]:
# hold every intercept at zero, learn the weights
model_fixed = nmo.glm.ClassifierPopulationGLM(
    classes=n_classes, fix_params=(None, np.zeros((n_neurons, n_classes)))
).fit(X, labels)

model_fixed.intercept_

Array([[0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.]], dtype=float32)

To hold the weights of a group of predictors instead, use a feature mask.

(neuron-specific-features-classifier)=
### Neuron-specific features

`feature_mask` selects the predictors of each neuron, as in the [`PopulationGLM`](nemos.glm.PopulationGLM). There is one difference: the mask has no class axis. Its shape is `(n_features, n_neurons)`. A feature reaches a neuron for all its classes, or not at all. You cannot use the mask for one class of a neuron only.

In [20]:
# drop the first 8 features from the predictors of the first neuron
feature_mask = np.ones((X.shape[1], n_neurons))
feature_mask[:8, 0] = 0

masked = nmo.glm.ClassifierPopulationGLM(
    classes=n_classes, feature_mask=feature_mask
)
masked.fit(X, labels)

# zero for every class of the first neuron
masked.coef_[:8, 0]

Array([[0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.]], dtype=float32)

## Examples

```{nemos-examples}
:model: ClassifierPopulationGLM
```